# Purpose of notebook
- Describe DetectPosi

In [1]:
import sys
sys.path.append('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Data_Collection\\Split_Page')

origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'

In [105]:
#Set up Environment
import os,cv2

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')
from Split import VertPart
from Read import Vision

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart

from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='

from google.cloud import vision
from google.cloud.vision_v1 import types

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [106]:
import json, os, cv2
import pandas as pd

Year,Showa='1950','25'
Quality='HQ'

df = pd.read_csv(origin+'Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)



# Step A. Set Page & Get Google OCR output

In [107]:
Page=20
file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/'+str(Year)+'/'+Quality+'/Page'+'{:03d}'.format(Page)+'.png'
img=cv2.imread(os.path.join(origin,file_path))

cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_Window", 1280, 1440)
cv2.imshow("Resized_Window",img)
cv2.waitKey(0)
cv2.destroyAllWindows()

#Convert to book page
BookPage=2*Page+16
Rows=df[(df['Page']==BookPage)]
NxRows=df[(df['Page']==BookPage+1)]
if len(Rows)==0:
    print('No Office at Right Side. Page'+str(BookPage))
if len(NxRows)==0:
    print('No Office at Left Side. Page'+str(BookPage+1))
        


In [108]:
texts=Vision(img,'zh',client)
textsCLOVA=Clova(origin,Page,api_url,secret_key,Year,Quality)

# Step B. Get Office Info

In [109]:
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
from Split import VertPart

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from Organize import Filter, ConvertDict
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart

file_path='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
path=os.path.join(origin,file_path)

HoriPoint=HoriPart(img,Page,texts)[0]

try:
    VertPoint=HH//3
except:
    print('Failed detecting Vertical Point')
    HH,WW=img.shape[:2]
    VertPoint=HH//3

##Right Page
BoxR=Filter(BookPage,texts,HoriPoint)
BoxR=ConvertDict(BoxR)

##Left Page
BoxL=Filter(BookPage+1,texts,HoriPoint)
BoxL=ConvertDict(BoxL)

Horizontal Line was not automatically detected... Defining line arbitrariry...


In [110]:
Dict={}
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from Organize import FilterBox
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from DetectOffice import DetectOffice
LocList=[]

#RightPage
OfficeList=Rows['Office'].tolist()
for Office in OfficeList:
    OfficeInfo=DetectOffice(BoxR, Office,VertPoint)
    if OfficeInfo==None:
        continue
    LocList.append(OfficeInfo)
    BoxR=FilterBox(BoxR,LocList,VertPoint)

#LeftPage
OfficeList=NxRows['Office'].tolist()
for Office in OfficeList:
    OfficeInfo=DetectOffice(BoxL, Office,VertPoint)
    if OfficeInfo==None:
        continue
    LocList.append(OfficeInfo)
    BoxL=FilterBox(BoxL,LocList,VertPoint)

Dict[Page]=LocList

# Main Code

-------------Input-------------

1. A dictionary with box of GV items corrensponding offices.
2. A dictionary of included offices with its starting location.

-------------Output-------------

A dictionary offices written in a quadrant with positions and their corresponding locations.

- **Description**
1. Check if office exists in page.

2. Cut texts into pieces based on quadrants.

3. If office exists 

    => Additionally crop texts into pieces.

4. Look for positions in each pieces.
    
5. Record detected positions

6. Adjust starting location according to position.

In [122]:
#Split quardrant into offices and detect Positions
from OrganizePosi import Split, SplitClova, Crop, CropClova
from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi

CrossWalk=pd.read_csv("C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
Positions=CrossWalk.tolist()
PosiDict,PosiDict_CLOVA={},{}
PosiDict['Pre'],PosiDict_CLOVA['Pre']=[],[]

- Crops a page into quadrant-wise pieces.

In [123]:
DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)

- "Split" detaches the quadrant of interest from the entire box and assigns them to office lists.

In [124]:
##UR
BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)

##LR
BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)

##UL
BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)

#LL
BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)

FinalPosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)

In [125]:
FinalPosiDict

{'Pre': [[]],
 '蠶業試験場': [[{'Position': '場長', 'Location': [2606, 238]}]],
 '八丈支場長': [[], [], [{'Position': '場長', 'Location': [2227, 255]}]],
 '農林工業指導所': [[{'Position': '所長', 'Location': [2628, 1126]}]],
 '農地課': [[{'Position': '課長', 'Location': [2365, 1128]}], []],
 '開拓係': [[{'Position': '係長', 'Location': [723, 286]}],
  [{'Position': '所長', 'Location': [1336, 1374]}]],
 '林務課': [[{'Position': '課長', 'Location': [1153, 1146]}]],
 '木材係': [[{'Position': '係長', 'Location': [1038, 1154]}]]}

- Refine PosiDict (Adjust starting position for '雇')

In [126]:
from DetectPosi import RefPosiDict
FinalPosiDict=RefPosiDict(texts,FinalPosiDict)
FinalPosiDict

IndexError: list index out of range

- Get new vertical point.

In [127]:
from Layout import RefineVert
VertPoint2=RefineVert(img,LocList,FinalPosiDict,VertPoint,HoriPoint)

In [128]:
VertPoint,VertPoint2

(715, 1106)

- Erase incorrectly detected positions.

In [129]:
from DetectPosi import RefPosiDict2
FinalPosiDict=RefPosiDict2(FinalPosiDict,VertPoint,VertPoint2)

# Show results

In [130]:
from ShowPosi import Show

In [132]:
for Office in list(FinalPosiDict.keys()):
    Show(FinalPosiDict,Office,img,VertPoint2,HoriPoint)

In [71]:
FinalPosiDict

{'Pre': [[]],
 '厚生課': [[{'Position': '所長', 'Location': [2465, 275]}]],
 '立川基地出張所': [[], [{'Position': '係長', 'Location': [2193, 1157]}]],
 '民政局': [[], []],
 '管理係': [[{'Position': '係長', 'Location': [1065, 241]}]],
 '指導係': [[{'Position': '係長', 'Location': [521, 232]}], []],
 '保護係': [[{'Position': '係長', 'Location': [1206, 1123]}]],
 '援護係': [[]]}